# 📊 Notebook 6: Evaluation Framework
## 4-Layer Evaluation + Memory A/B Testing + Latency Benchmarks

This notebook runs the complete evaluation suite:

| Layer | Metrics | Method |
|---|---|---|
| **1. Retrieval** | Hit Rate, MRR, Precision@k, Recall@k | Automated |
| **2. Lexical** | Groundedness, Hallucination Rate, Context Utilization | Automated |
| **3. LLM-as-Judge** | Context Relevance, Faithfulness, Correctness | LLM scoring |
| **4. Memory** | Memory Recall, Personalization, A/B Test | LLM scoring |

---

In [ ]:
import sys
sys.path.insert(0, '..')

from src.eval.metrics import (
    compute_groundedness, compute_hallucination_rate,
    compute_context_utilization, compute_answer_relevance,
    compute_factual_consistency
)
from src.eval.test_cases import TEST_CASES

print('✅ Evaluation modules loaded!')
print(f'📋 Total test cases: {len(TEST_CASES)}')

## Step 1: Inspect Test Dataset

Our test suite has 30+ curated cases across 5 categories.

In [ ]:
# Categorize test cases
from collections import Counter

categories = Counter(tc.get('category', tc.get('route', 'unknown')) for tc in TEST_CASES)

print('📊 Test Cases by Category:')
for cat, count in categories.most_common():
    print(f'   {cat:<25} {count:>3} cases')

print(f'\n--- Sample Test Cases ---')
for tc in TEST_CASES[:5]:
    print(f'\n  Q: {tc["question"]}')
    print(f'  Category: {tc.get("category", tc.get("route", "unknown"))}')
    if 'ground_truth' in tc:
        print(f'  Ground Truth: {tc["ground_truth"][:100]}...')

## Step 2: Lexical Metrics Demo (Free, No LLM Required)

These metrics use word overlap — completely free and instant.

In [ ]:
# Demo with synthetic examples
test_examples = [
    {
        'name': 'Well-grounded answer',
        'question': 'What is machine learning?',
        'context': 'Machine learning is a subset of artificial intelligence that uses statistical methods to enable systems to learn from data and improve their performance.',
        'answer': 'Machine learning is a subset of artificial intelligence that enables systems to learn from data using statistical methods.',
        'ground_truth': 'Machine learning is a branch of AI that uses data to learn and improve.',
    },
    {
        'name': 'Hallucinated answer',
        'question': 'What is machine learning?',
        'context': 'Machine learning is a subset of artificial intelligence.',
        'answer': 'Machine learning was invented by Elon Musk in 2015 and uses quantum computing to train models on Mars.',
        'ground_truth': 'Machine learning is a branch of AI.',
    },
    {
        'name': 'Partially grounded',
        'question': 'What is deep learning?',
        'context': 'Deep learning uses neural networks with many layers to model complex patterns.',
        'answer': 'Deep learning uses neural networks with many layers. It was popularized by Geoffrey Hinton and has transformed computer vision.',
        'ground_truth': 'Deep learning uses multi-layer neural networks for pattern recognition.',
    },
]

print(f'{"Example":<25} {"Groundedness":>13} {"Hallucination":>14} {"Ctx Util":>10} {"Relevance":>10} {"Factual":>10}')
print('-' * 85)

for ex in test_examples:
    g = compute_groundedness(ex['answer'], ex['context'])
    h = compute_hallucination_rate(ex['answer'], ex['context'])
    u = compute_context_utilization(ex['answer'], ex['context'])
    r = compute_answer_relevance(ex['answer'], ex['question'])
    f = compute_factual_consistency(ex['answer'], ex['ground_truth'])
    
    print(f'{ex["name"]:<25} {g:>12.1%} {h:>13.1%} {u:>9.1%} {r:>9.1%} {f:>9.1%}')

## Step 3: Visualize Evaluation Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Compute metrics for all examples
metrics_names = ['Groundedness', 'Hallucination', 'Context Util', 'Relevance', 'Factual']
example_names = [ex['name'] for ex in test_examples]

data = []
for ex in test_examples:
    row = [
        compute_groundedness(ex['answer'], ex['context']),
        compute_hallucination_rate(ex['answer'], ex['context']),
        compute_context_utilization(ex['answer'], ex['context']),
        compute_answer_relevance(ex['answer'], ex['question']),
        compute_factual_consistency(ex['answer'], ex['ground_truth']),
    ]
    data.append(row)

data = np.array(data)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(metrics_names))
width = 0.25
colors = ['#55A868', '#E24A33', '#4C72B0']

for i, (name, color) in enumerate(zip(example_names, colors)):
    ax.bar(x + i * width, data[i], width, label=name, color=color, alpha=0.85, edgecolor='white')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Evaluation Metrics Comparison', fontsize=16, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics_names, fontsize=11)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print('\n💡 Key Observations:')
print('  • The well-grounded answer scores high on groundedness and low on hallucination')
print('  • The hallucinated answer scores very low — our metrics catch it!')
print('  • Partially grounded answers show moderate scores across metrics')

## Step 4: Full Evaluation Suite

> **Note:** Running the full evaluation requires an LLM provider and ingested data.

In [ ]:
# Full evaluation (requires LLM + ingested data)
try:
    from src.eval.evaluator import RAGEvaluator
    evaluator = RAGEvaluator()
    results = evaluator.run()
    
    print('📊 Full Evaluation Results:')
    for metric, value in results.items():
        if isinstance(value, float):
            print(f'   {metric:<30} {value:.3f}')
        else:
            print(f'   {metric:<30} {value}')
except Exception as e:
    print('⚠️ Full evaluation requires:')
    print('   1. LLM provider configured in .env')
    print('   2. Data ingested via: python scripts/ingest.py --file urls.txt')
    print(f'   Error: {e}')
    print()
    print('✅ The lexical metrics above demonstrate the framework works!')
    print('   Configure your .env and run the full suite for complete results.')